In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                             f1_score, precision_score, roc_auc_score, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay,
                             average_precision_score)
from sklearn.utils import resample

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
warnings.filterwarnings('ignore')
import joblib

In [3]:
import numpy as np
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             f1_score, confusion_matrix)

def evalueaza_model_cu_prag(model, X_test, y_test, nume_set, prag):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred_custom = (y_proba >= prag).astype(int)
    acc = accuracy_score(y_test, y_pred_custom)
    auc_roc = roc_auc_score(y_test, y_proba)
    auprc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred_custom)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"=== Rezultate pentru: {nume_set} ===")
    print(f"Prag (Threshold) utilizat: {prag:.4f}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUPRC:       {auprc:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"Sensitivity: {sens:.4f}")
    print(f"Specificity: {spec:.4f}")
    print("=====================================\n")

In [5]:
data = pd.read_excel('/content/dataset.xlsx')
print("Shape: ", data.shape)

features_df    = pd.read_csv('/content/FINAL_35_features_selected.csv')
FINAL_FEATURES = features_df['feature'].tolist()
N_FEATURES     = len(FINAL_FEATURES)

X = data[FINAL_FEATURES]
y = data['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Shape:  (886, 61)


In [6]:
features_df_35 = pd.read_csv('/content/FINAL_35_features_selected.csv')

if 'feature' in features_df_35.columns:
    FINAL_FEATURES_35 = features_df_35['feature'].tolist()
else:
    FINAL_FEATURES_35 = [col for col in features_df_35.columns if col != 'label']

X_35 = data[FINAL_FEATURES_35]

In [11]:
features_df_24 = pd.read_csv('/content/24_Goldstein_features.csv')

if 'feature' in features_df_24.columns:
    FINAL_FEATURES_24 = features_df_24['feature'].tolist()
else:
    FINAL_FEATURES_24 = [col for col in features_df_24.columns if col != 'label']

X_24 = data[FINAL_FEATURES_24]

In [17]:
X_35 = data[FINAL_FEATURES_35]
X_train_35, X_test_35, y_train_35, y_test_35 = train_test_split(
    X_35, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Distributions:")
print(f"X_test shape:  {X_test_35.shape} ({X_test_35.shape[0]} paciente, {X_test_35.shape[1]} simptome)\n")
date_joblib = joblib.load('/content/rf_model_youden.joblib')
print(date_joblib.keys())
model_35_extras = date_joblib['model']
prag_youden = date_joblib['threshold']
evalueaza_model_cu_prag(
    model=model_35_extras,
    X_test=X_test_35,
    y_test=y_test_35,
    nume_set="35 Features (Set TEST 20%)",
    prag=prag_youden
)

Distributions:
X_test shape:  (178, 35) (178 paciente, 35 simptome)

dict_keys(['model', 'threshold', 'features'])
=== Rezultate pentru: 35 Features (Set TEST 20%) ===
Prag (Threshold) utilizat: 0.5377
Accuracy:    0.9326
AUC-ROC:     0.9762
AUPRC:       0.9807
F1-Score:    0.9355
Sensitivity: 0.9158
Specificity: 0.9518



In [18]:
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

X_24 = data[FINAL_FEATURES_24]
X_train_24, X_test_24, y_train_24, y_test_24 = train_test_split(
    X_24, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Distributions:")
print(f"X_test shape:  {X_test_24.shape} ({X_test_24.shape[0]} paciente, {X_test_24.shape[1]} simptome)\n")
date_joblib = joblib.load('/content/rf_model_youden.joblib')
prag_youden = date_joblib['threshold']
print(f"-> Prag Youden extras din .joblib: {prag_youden:.4f}\n")
print("ANTRENARE MODEL: RANDOM FOREST")
best_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_train_24, y_train_24)
cv_outer_rf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rf = cross_val_score(
    best_rf,
    X_train_24,
    y_train_24,
    cv=cv_outer_rf,
    scoring='roc_auc',
    n_jobs=1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}\n")
evalueaza_model_cu_prag(
    model=best_rf,
    X_test=X_test_24,
    y_test=y_test_24,
    nume_set="24 Features (Model Nou antrenat + Prag Joblib)",
    prag=prag_youden
)

Distributions:
X_test shape:  (178, 24) (178 paciente, 24 simptome)

-> Prag Youden extras din .joblib: 0.5377

ANTRENARE MODEL: RANDOM FOREST
10-Fold CV (Train) AUC-ROC: 0.9742 ± 0.0116

=== Rezultate pentru: 24 Features (Model Nou antrenat + Prag Joblib) ===
Prag (Threshold) utilizat: 0.5377
Accuracy:    0.8876
AUC-ROC:     0.9773
AUPRC:       0.9824
F1-Score:    0.8958
Sensitivity: 0.9053
Specificity: 0.8675



In [20]:
X_24 = data[FINAL_FEATURES_24]
X_train_24, X_test_24, y_train_24, y_test_24 = train_test_split(
    X_24, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Distributions:")
print(f"X_test shape:  {X_test_24.shape} ({X_test_24.shape[0]} paciente, {X_test_24.shape[1]} simptome)\n")
print("TRAIN MODEL: RANDOM FOREST")
best_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
cv_outer_rf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rf = cross_val_score(
    best_rf,
    X_train_24,
    y_train_24,
    cv=cv_outer_rf,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}\n")
best_rf.fit(X_train_24, y_train_24)
y_proba_train_oof = cross_val_predict(
    best_rf,
    X_train_24,
    y_train_24,
    cv=cv_outer_rf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

fpr, tpr, thresholds = roc_curve(y_train_24, y_proba_train_oof)
youden_j = tpr - fpr
best_threshold_idx = np.argmax(youden_j)
prag_youden_nou = thresholds[best_threshold_idx]
print(f"-> New Youden Threshold: {prag_youden_nou:.4f}\n")

print("="*60)
print("EVALUARE: PRAG IMPLICIT (0.50)")
print("="*60)
evalueaza_model_cu_prag(
    model=best_rf,
    X_test=X_test_24,
    y_test=y_test_24,
    nume_set="24 Features (Default Threshold 0.50)",
    prag=0.50
)

evalueaza_model_cu_prag(
    model=best_rf,
    X_test=X_test_24,
    y_test=y_test_24,
    nume_set="24 Features (New Youden Threshold)",
    prag=prag_youden_nou
)

Distributions:
X_test shape:  (178, 24) (178 paciente, 24 simptome)

TRAIN MODEL: RANDOM FOREST
10-Fold CV (Train) AUC-ROC: 0.9742 ± 0.0116

-> New Youden Threshold: 0.6100

EVALUARE: PRAG IMPLICIT (0.50)
=== Rezultate pentru: 24 Features (Default Threshold 0.50) ===
Prag (Threshold) utilizat: 0.5000
Accuracy:    0.8820
AUC-ROC:     0.9773
AUPRC:       0.9824
F1-Score:    0.8923
Sensitivity: 0.9158
Specificity: 0.8434

=== Rezultate pentru: 24 Features (New Youden Threshold) ===
Prag (Threshold) utilizat: 0.6100
Accuracy:    0.8989
AUC-ROC:     0.9773
AUPRC:       0.9824
F1-Score:    0.9053
Sensitivity: 0.9053
Specificity: 0.8916



**DCA RELATED WORK**

In [24]:
def calculate_single_nb(y_true, y_proba, p_t):
    n = len(y_true)
    y_true_arr = np.asarray(y_true)
    y_proba_arr = np.asarray(y_proba)

    if p_t == 1.0:
        p_t = 0.999

    y_pred = (y_proba_arr >= p_t).astype(int)

    tp = np.sum((y_pred == 1) & (y_true_arr == 1))
    fp = np.sum((y_pred == 1) & (y_true_arr == 0))

    nb = (tp - fp * (p_t / (1 - p_t))) / n
    return nb

In [26]:
y_proba_test_24 = best_rf.predict_proba(X_test_24)[:, 1]
scenarios = {
    "Screening Scenario (Threshold = 20%)": 0.20,
    "Standard Scenario (Threshold = 50%)": 0.50,
    "Confirmation Scenario (Threshold = 80%)": 0.80
}

for name, threshold_val in scenarios.items():
    nb_model = calculate_single_nb(y_test_24, y_proba_test_24, threshold_val)
    y_proba_all = np.ones(len(y_test_24))
    nb_all = calculate_single_nb(y_test_24, y_proba_all, threshold_val)
    delta_all = nb_model - nb_all
    print(f"   Active Decision Threshold (p_t): {threshold_val:.2f} ({threshold_val * 100:.0f}%)")
    print(f"   Net Benefit Model (24 Feat.):    {nb_model:.4f}")
    print(f"   Net Benefit 'Treat All':         {nb_all:.4f}")

    if threshold_val == 0.20:
        print(f"   Improvement vs. 'Treat All':     +{delta_all:.4f}")
    elif threshold_val == 0.50:
        print(f"   Improvement vs. 'Treat All':     +{delta_all:.4f}")
        additional_patients = delta_all * 100
        print(f"   Equivalent Clinical Lift:        +{additional_patients:.1f} true positive patients detected per 100 tests (no false alarms)")
    elif threshold_val == 0.80:
        print(f"   - Improvement vs. 'Treat None':    +{nb_model:.4f}")
    print("-"*80)

   Active Decision Threshold (p_t): 0.20 (20%)
   Net Benefit Model (24 Feat.):    0.4902
   Net Benefit 'Treat All':         0.4171
   Improvement vs. 'Treat All':     +0.0730
--------------------------------------------------------------------------------
   Active Decision Threshold (p_t): 0.50 (50%)
   Net Benefit Model (24 Feat.):    0.4157
   Net Benefit 'Treat All':         0.0674
   Improvement vs. 'Treat All':     +0.3483
   Equivalent Clinical Lift:        +34.8 true positive patients detected per 100 tests (no false alarms)
--------------------------------------------------------------------------------
   Active Decision Threshold (p_t): 0.80 (80%)
   Net Benefit Model (24 Feat.):    0.4270
   Net Benefit 'Treat All':         -1.3315
   - Improvement vs. 'Treat None':    +0.4270
--------------------------------------------------------------------------------
